# 📦 Demand Forecasting & Inventory Optimization Engine
## Milestone 1: Data Collection & Exploratory Data Analysis (EDA)

---

**👥 Team Members:**
- Mark Mohsen Nasry
- Fady Milad Shahat
- Ahmed Arafa Hafez
- Eman Ahmed Sayed
- Sama Ashraf Mahmoud
- Hadeer Mahmoud Abdelsalam

**📅 Date:** February 2026  
**🎯 Objective:** Collect, clean, and explore the demand forecasting dataset to understand patterns, trends, and anomalies before modeling.

---

### 📋 Table of Contents
1. [Environment Setup](#1.-Environment-Setup)
2. [Data Loading](#2.-Data-Loading)
3. [Data Overview & Quality Check](#3.-Data-Overview-&-Quality-Check)
4. [Data Cleaning & Preprocessing](#4.-Data-Cleaning-&-Preprocessing)
5. [Univariate Analysis](#5.-Univariate-Analysis)
6. [Time-Series Analysis](#6.-Time-Series-Analysis)
7. [Correlation & Multivariate Analysis](#7.-Correlation-&-Multivariate-Analysis)
8. [Key Findings & Conclusions](#8.-Key-Findings-&-Conclusions)
9. [Save Processed Data](#9.-Save-Processed-Data)

---
## 1. Environment Setup

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import os
from pathlib import Path

# ── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Stats ────────────────────────────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Settings ─────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)

# ── Plotting Style ────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGSIZE_WIDE = (16, 5)
FIGSIZE_SQ   = (10, 6)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
RAW_DATA_DIR  = PROJECT_ROOT / 'data' / 'raw'
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
PROC_DATA_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Environment ready!')
print(f'   📁 Raw data dir   : {RAW_DATA_DIR}')
print(f'   📁 Processed dir  : {PROC_DATA_DIR}')

---
## 2. Data Loading

> **📝 TODO (Team):** Update the `DATA_FILE` variable below with your actual dataset filename.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔧 CONFIGURE: Change this to your actual data file
# ─────────────────────────────────────────────────────────────────────────────
DATA_FILE = 'your_dataset.csv'   # <-- UPDATE THIS
DATE_COL  = 'date'               # <-- UPDATE: name of your date column
TARGET_COL = 'demand'            # <-- UPDATE: name of your target/sales column
# ─────────────────────────────────────────────────────────────────────────────

DATA_PATH = RAW_DATA_DIR / DATA_FILE

# Load based on file extension
ext = DATA_PATH.suffix.lower()
if ext == '.csv':
    df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
elif ext in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH, parse_dates=[DATE_COL])
elif ext == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'Unsupported file format: {ext}')

print(f'✅ Dataset loaded: {DATA_FILE}')
print(f'   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Memory : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
df.head(10)

---
## 3. Data Overview & Quality Check

In [ ]:
# ── Schema ───────────────────────────────────────────────────────────────────
print('='*60)
print('📊 DATASET SCHEMA')
print('='*60)
df.info()
print()

# ── Date range ────────────────────────────────────────────────────────────────
print('='*60)
print('📅 DATE RANGE')
print('='*60)
print(f'  Start : {df[DATE_COL].min()}')
print(f'  End   : {df[DATE_COL].max()}')
print(f'  Span  : {(df[DATE_COL].max() - df[DATE_COL].min()).days} days')

In [ ]:
# ── Missing Values ────────────────────────────────────────────────────────────
print('='*60)
print('🔍 MISSING VALUES')
print('='*60)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('  ✅ No missing values found!')
else:
    print(missing_df)

    # Visualize
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing_df)*0.5)))
    missing_df['Missing %'].plot(kind='barh', ax=ax, color='#e74c3c')
    ax.set_title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Missing %')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
n_dups = df.duplicated().sum()
print(f'🔁 Duplicate rows: {n_dups:,}')
if n_dups > 0:
    print('   ⚠️  Duplicates found! Review before dropping.')
    display(df[df.duplicated(keep=False)].head(10))

# ── Basic Stats ───────────────────────────────────────────────────────────────
print('\n📈 DESCRIPTIVE STATISTICS')
df.describe(include='all').T

---
## 4. Data Cleaning & Preprocessing

In [ ]:
df_clean = df.copy()

# ── 1. Drop exact duplicates ──────────────────────────────────────────────────
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'  Removed duplicates : {before - len(df_clean):,} rows')

# ── 2. Parse & sort by date ───────────────────────────────────────────────────
df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL])
df_clean = df_clean.sort_values(DATE_COL).reset_index(drop=True)
print(f'  Sorted by date ✅')

# ── 3. Handle missing values ──────────────────────────────────────────────────
# 🔧 TODO: Choose the right strategy for your dataset
# Option A: Forward-fill (good for time series)
# df_clean[TARGET_COL] = df_clean[TARGET_COL].fillna(method='ffill')

# Option B: Interpolate (smooth)
# df_clean[TARGET_COL] = df_clean[TARGET_COL].interpolate(method='linear')

# Option C: Drop rows with missing target
df_clean = df_clean.dropna(subset=[TARGET_COL])
print(f'  After NaN removal  : {len(df_clean):,} rows')

# ── 4. Remove negative demand values ─────────────────────────────────────────
neg = (df_clean[TARGET_COL] < 0).sum()
df_clean = df_clean[df_clean[TARGET_COL] >= 0]
print(f'  Removed negative values : {neg:,}')

# ── 5. Feature Engineering ────────────────────────────────────────────────────
df_clean['year']       = df_clean[DATE_COL].dt.year
df_clean['month']      = df_clean[DATE_COL].dt.month
df_clean['week']       = df_clean[DATE_COL].dt.isocalendar().week.astype(int)
df_clean['dayofweek']  = df_clean[DATE_COL].dt.dayofweek
df_clean['quarter']    = df_clean[DATE_COL].dt.quarter
df_clean['is_weekend'] = (df_clean['dayofweek'] >= 5).astype(int)

print(f'\n✅ Cleaning done! Final shape: {df_clean.shape}')
df_clean.head()

---
## 5. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Distribution of {TARGET_COL}', fontsize=16, fontweight='bold', y=1.02)

# Histogram
axes[0].hist(df_clean[TARGET_COL], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Histogram')
axes[0].set_xlabel(TARGET_COL)

# Box plot
axes[1].boxplot(df_clean[TARGET_COL], patch_artist=True,
                boxprops=dict(facecolor='#2ecc71', alpha=0.7))
axes[1].set_title('Box Plot (Outlier Check)')
axes[1].set_xlabel(TARGET_COL)

# Log-scale histogram
axes[2].hist(np.log1p(df_clean[TARGET_COL]), bins=50, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[2].set_title('Log(1 + demand) Histogram')
axes[2].set_xlabel(f'log(1 + {TARGET_COL})')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Skewness : {df_clean[TARGET_COL].skew():.4f}')
print(f'  Kurtosis : {df_clean[TARGET_COL].kurt():.4f}')

In [ ]:
# ── Monthly & Day-of-Week Patterns ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
day_names   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

monthly = df_clean.groupby('month')[TARGET_COL].mean()
daily   = df_clean.groupby('dayofweek')[TARGET_COL].mean()

axes[0].bar(monthly.index, monthly.values, color='#e67e22', edgecolor='white')
axes[0].set_title('Average Demand by Month', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Avg Demand')

axes[1].bar(daily.index, daily.values, color='#1abc9c', edgecolor='white')
axes[1].set_title('Average Demand by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_names)
axes[1].set_ylabel('Avg Demand')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_seasonal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Time-Series Analysis

In [ ]:
# ── Full Time-Series Plot ─────────────────────────────────────────────────────
ts = df_clean.set_index(DATE_COL)[TARGET_COL]

fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
ax.plot(ts.index, ts.values, linewidth=0.8, color='#2c3e50', alpha=0.7, label='Actual')

# 30-day rolling average
ts_rolling = ts.rolling(window=30, min_periods=1).mean()
ax.plot(ts_rolling.index, ts_rolling.values, linewidth=2, color='#e74c3c', label='30-day MA')

ax.set_title(f'{TARGET_COL} Over Time', fontsize=15, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel(TARGET_COL)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Seasonal Decomposition ───────────────────────────────────────────────────
# 🔧 Adjust 'period' based on your data frequency (7=weekly, 12=monthly, 365=yearly)
PERIOD = 7   # <-- UPDATE THIS

# Resample to daily if needed
ts_daily = ts.resample('D').sum().fillna(0)

decomp = seasonal_decompose(ts_daily, model='additive', period=PERIOD)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
colors = ['#2c3e50', '#3498db', '#e74c3c', '#27ae60']
labels = ['Original', 'Trend', 'Seasonal', 'Residual']

for ax, data, color, label in zip(axes,
    [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid],
    colors, labels):
    ax.plot(data, color=color, linewidth=0.9)
    ax.set_ylabel(label, fontsize=11)
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

axes[0].set_title('Seasonal Decomposition (Additive)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ACF & PACF ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(ts_daily.dropna(),  lags=50, ax=axes[0], color='#3498db')
plot_pacf(ts_daily.dropna(), lags=50, ax=axes[1], color='#e74c3c')

axes[0].set_title('Autocorrelation Function (ACF)',  fontsize=13, fontweight='bold')
axes[1].set_title('Partial Autocorrelation (PACF)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Correlation & Multivariate Analysis

In [ ]:
# Select only numeric columns
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

corr = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols)), max(6, len(numeric_cols)*0.8)))
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
print(f'\n🎯 Top correlations with [{TARGET_COL}]:')
print(corr[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False).head(10))

---
## 8. Key Findings & Conclusions

> **📝 TODO (Team):** Fill in your findings based on the analysis above.

### ✅ Summary

| Finding | Details |
|---|---|
| Dataset Size | *rows × cols* |
| Date Range | *start → end* |
| Missing Data | *X% missing, handled by …* |
| Trend | *Upward / Downward / Stable* |
| Seasonality | *Weekly / Monthly / Yearly* |
| Outliers | *X outliers detected, strategy = …* |
| Top Features | *feature1, feature2, …* |

### 🚀 Next Steps (Milestone 2)
- [ ] Feature selection based on correlation
- [ ] Train/Validation/Test split strategy
- [ ] Baseline model: Naive / SARIMA
- [ ] Advanced models: Prophet, XGBoost, LSTM

---
## 9. Save Processed Data

In [ ]:
output_path = PROC_DATA_DIR / 'demand_cleaned.parquet'
df_clean.to_parquet(output_path, index=False)
print(f'✅ Cleaned dataset saved to:')
print(f'   {output_path}')
print(f'   Shape: {df_clean.shape}')